# Deepchecks: validación de datos

Este notebook está pensado para **probar solo checks de validación de datos** en Deepchecks.

No usa modelos.

## Qué incluye
- un dataset fake tabular
- una columna objetivo para poder probar `ClassImbalance`
- varios datasets "corruptos" para provocar distintos problemas
- **7 checks importantes** de data integrity:
  1. `PercentOfNulls`
  2. `MixedNulls`
  3. `MixedDataTypes`
  4. `SpecialCharacters`
  5. `StringMismatch`
  6. `DataDuplicates`
  7. `ClassImbalance`

## 1. Instalación

Si ya tienes Deepchecks instalado en un entorno compatible, puedes saltarte esta celda.

In [1]:
# Descomenta solo si necesitas instalar Deepchecks
!pip install "deepchecks" -q


import numpy as np
np.Inf = np.inf
np.NINF = -np.inf
np.PINF = np.inf
np.NaN = np.nan

from deepchecks.tabular.checks import MixedNulls

print("✅ Deepchecks instalado correctamente. Si acabas de instalarlo, reinicia el runtime.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.3/357.3 kB 28.0 MB/s eta 0:00:00
✅ Deepchecks instalado correctamente. Si acabas de instalarlo, reinicia el runtime.


## 2. Imports

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from deepchecks.tabular import Dataset
from deepchecks.tabular.checks import (
    PercentOfNulls,
    MixedNulls,
    MixedDataTypes,
    SpecialCharacters,
    StringMismatch,
    DataDuplicates,
    ClassImbalance
)

## 3. Crear dataset fake base

Generamos:
- variables numéricas
- variables categóricas
- una columna de texto corto
- una etiqueta binaria `target`

In [3]:
np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "edad": np.random.randint(18, 75, size=n),
    "ingresos": np.random.normal(35000, 12000, size=n).round(2),
    "score": np.random.normal(600, 80, size=n).round(1),
    "categoria": np.random.choice(["A", "B", "C"], size=n, p=[0.5, 0.3, 0.2]),
    "ciudad": np.random.choice(["Madrid", "Barcelona", "Valencia"], size=n, p=[0.45, 0.35, 0.20]),
    "comentario": np.random.choice(["ok", "normal", "revisar", "sin problema"], size=n)
})

# target binario con algo de relación con variables numéricas
logit = (
    0.03 * (df["edad"] - 40)
    + 0.00005 * (df["ingresos"] - 35000)
    + 0.01 * (df["score"] - 600)
    + (df["categoria"] == "A").astype(int) * 0.3
    - 0.25
)

prob = 1 / (1 + np.exp(-logit))
df["target"] = (np.random.rand(n) < prob).astype(int)

df = df.reset_index(drop=True)

display(df.head())
print(df.shape)
print(df["target"].value_counts(normalize=True).sort_index())

,edad,ingresos,score,categoria,ciudad,comentario,target
0,56,10190.69,614.5,B,Barcelona,normal,1
1,69,33930.56,496.3,A,Barcelona,normal,0
2,46,19346.37,632.0,B,Madrid,sin problema,0
3,32,43036.07,547.9,A,Madrid,ok,1
4,60,39399.18,557.7,A,Barcelona,revisar,1


(1000, 7)
target
0    0.475
1    0.525
Name: proportion, dtype: float64


## 4. Crear datasets modificados para cada check

Cada bloque fabrica un problema distinto en los datos.

In [4]:
# 4.1 PercentOfNulls: nulos reales
df_null_pct = df.copy()
null_idx_edad = np.random.choice(df_null_pct.index, size=120, replace=False)
null_idx_ing = np.random.choice(df_null_pct.index, size=80, replace=False)
df_null_pct.loc[null_idx_edad, "edad"] = np.nan
df_null_pct.loc[null_idx_ing, "ingresos"] = np.nan

# 4.2 MixedNulls: varias representaciones de nulos
df_mixed_nulls = df.copy()
df_mixed_nulls["comentario"] = df_mixed_nulls["comentario"].astype(object)
df_mixed_nulls.loc[100:110, "comentario"] = "N/A"
df_mixed_nulls.loc[111:120, "comentario"] = "none"
df_mixed_nulls.loc[121:125, "comentario"] = ""

# 4.3 MixedDataTypes: mezclar números y strings
df_mixed_types = df.copy()
df_mixed_types["score_mixto"] = df_mixed_types["score"].astype(object)
df_mixed_types.loc[0:50, "score_mixto"] = df_mixed_types.loc[0:50, "score"].astype(str)
df_mixed_types.loc[200:240, "score_mixto"] = "texto_invalido"

# 4.4 SpecialCharacters: columna textual con símbolos raros
df_special_chars = df.copy()
df_special_chars["comentario_raro"] = "normal"
df_special_chars["comentario_raro"] = df_special_chars["comentario_raro"].astype(object)
df_special_chars.loc[0:35, "comentario_raro"] = "!@#$%^&*()_+{}|:<>?"

# 4.5 StringMismatch: variantes de strings parecidos
df_string_mismatch = df.copy()
df_string_mismatch["ciudad"] = df_string_mismatch["ciudad"].astype(object)
df_string_mismatch.loc[0:20, "ciudad"] = "madrid"
df_string_mismatch.loc[21:35, "ciudad"] = "Madrid "
df_string_mismatch.loc[36:45, "ciudad"] = "MADRID"

# 4.6 DataDuplicates: duplicar filas
df_duplicates = pd.concat([df, df.iloc[:25]], ignore_index=True)

# 4.7 ClassImbalance: hacer la clase 1 muy minoritaria
df_imbalance = df.copy()
idx_class1 = df_imbalance[df_imbalance["target"] == 1].index[40:]
df_imbalance.loc[idx_class1, "target"] = 0

## 5. Crear objetos `Dataset` de Deepchecks

In [5]:
cat_features_base = ["categoria", "ciudad", "comentario"]

ds_null_pct = Dataset(df_null_pct, label="target", cat_features=cat_features_base)
ds_mixed_nulls = Dataset(df_mixed_nulls, label="target", cat_features=cat_features_base)
ds_mixed_types = Dataset(df_mixed_types, label="target", cat_features=cat_features_base + ["score_mixto"])
ds_special_chars = Dataset(df_special_chars, label="target", cat_features=cat_features_base + ["comentario_raro"])
ds_string_mismatch = Dataset(df_string_mismatch, label="target", cat_features=cat_features_base)
ds_duplicates = Dataset(df_duplicates, label="target", cat_features=cat_features_base)
ds_imbalance = Dataset(df_imbalance, label="target", cat_features=cat_features_base)

## 6. Helper para ejecutar checks sin parar el notebook

Así, si uno falla por entorno o versión, el resto sigue ejecutándose.

In [6]:
def run_check(name, check_obj, dataset):
    print("=" * 90)
    print(name)
    print("=" * 90)
    try:
        result = check_obj.run(dataset)
        display(result)
        try:
            result.show()
        except Exception as e:
            print("No se pudo ejecutar result.show():", repr(e))
        return result
    except Exception as e:
        print("ERROR:", repr(e))
        return None

## 7. Ejecutar los 7 checks más importantes de validación de datos

In [7]:
run_check("1) PercentOfNulls", PercentOfNulls(), ds_null_pct)
run_check("2) MixedNulls", MixedNulls(), ds_mixed_nulls)
run_check("3) MixedDataTypes", MixedDataTypes(), ds_mixed_types)
run_check("4) SpecialCharacters", SpecialCharacters(), ds_special_chars)
run_check("5) StringMismatch", StringMismatch(), ds_string_mismatch)
run_check("6) DataDuplicates", DataDuplicates(), ds_duplicates)
run_check("7) ClassImbalance", ClassImbalance(), ds_imbalance)

1) PercentOfNulls


<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 PercentOfNulls

<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 PercentOfNulls

2) MixedNulls


<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Mixed Nulls

...

<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Data Duplicates

7) ClassImbalance


<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Class Imbalance

<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Class Imbalance

<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Class Imbalance

## 8. Suite completa de integridad

Si quieres una revisión más amplia sobre un único dataset, Deepchecks también tiene una suite `data_integrity()`.



In [9]:
from deepchecks.tabular.suites import data_integrity
suite = data_integrity()
suite_result = suite.run(ds_null_pct)
suite_result

<!--
 ~ ----------------------------------------------------------------------------
 ~ Copyright (C) 2021-2023 Deepchecks (https://www.deepchecks.com)
 ~
 ~ This file is part of Deepchecks.
 ~ Deepchecks is distributed under the terms of the GNU Affero General
 ~ Public License (version 3 or later).
 ~ You should have received a copy of the GNU Affero General Public License
 ~ along with Deepchecks. If not, see .
 ~ ----------------------------------------------------------------------------
 ~
-->
<!DOCTYPE html>
 
 
 
 Data Integrity Suite

## 9. Resumen

Este notebook te deja una base sencilla para enseñar o probar Deepchecks sin entrar todavía en modelos.

### Checks incluidos
- `PercentOfNulls`
- `MixedNulls`
- `MixedDataTypes`
- `SpecialCharacters`
- `StringMismatch`
- `DataDuplicates`
- `ClassImbalance`

